# SentinelAI — 04 Unsupervised Anomaly Detection (No Label Leakage)

الخوارزميات:
1. K-Means
2. DBSCAN
3. Isolation Forest

قواعد هذه النسخة:
- لا تستخدم `Label` في `fit`.
- لا نضبط `contamination` من نسبة الهجمات الحقيقية.
- K-Means يحدد anomaly cluster بالافتراض غير الخاضع للإشراف: **أصغر cluster**.
- `Label` يستخدم **بعد** التوقع فقط لحساب المقاييس.

In [1]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import joblib

from sklearn.cluster import KMeans, DBSCAN
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

cwd = Path.cwd()
BASE_DIR = cwd.parent if cwd.name == "notebooks" else cwd
P = BASE_DIR / "data" / "processed"
M = BASE_DIR / "models"
M.mkdir(exist_ok=True)

X_train = np.load(P / "X_train.npy", mmap_mode="r")
X_val = np.load(P / "X_val.npy", mmap_mode="r")
y_train = np.load(P / "y_train.npy", mmap_mode="r")
y_val = np.load(P / "y_val.npy", mmap_mode="r")

def as_attack(y):
    return (np.asarray(y) != 1).astype(int)

def bin_metrics(name, y_true, y_pred, note):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    return {
        "Algorithm": name,
        "Task": "binary_anomaly_detection",
        "Evaluation_Split": "validation",
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": p,
        "Recall": r,
        "F1": f1,
        "Label_Use": "post_hoc_evaluation_only",
        "Method": note,
    }

rows = []
print("Train:", X_train.shape, "Validation:", X_val.shape)

Train: (908327, 77) Validation: (113541, 77)


In [2]:
# Label-free training sample.
n_km = min(100_000, len(X_train))
idx_km = rng.choice(len(X_train), size=n_km, replace=False)
X_km = np.asarray(X_train[idx_km])

start = time.time()
kmeans = KMeans(n_clusters=2, n_init=10, random_state=RANDOM_STATE)
kmeans.fit(X_km)

cluster_sizes = np.bincount(kmeans.labels_)
anomaly_cluster = int(np.argmin(cluster_sizes))  # no labels used

val_clusters = kmeans.predict(np.asarray(X_val))
km_attack = (val_clusters == anomaly_cluster).astype(int)

rows.append(bin_metrics(
    "K-Means", as_attack(y_val), km_attack,
    f"minority_cluster={anomaly_cluster}; train_cluster_sizes={cluster_sizes.tolist()}"
))
joblib.dump(kmeans, M / "kmeans.joblib")
print("K-Means cluster sizes:", cluster_sizes, "anomaly cluster:", anomaly_cluster)

K-Means cluster sizes: [71802 28198] anomaly cluster: 1


In [3]:
# Isolation Forest: no true attack ratio is supplied.
n_iso = min(150_000, len(X_train))
idx_iso = rng.choice(len(X_train), size=n_iso, replace=False)
X_iso = np.asarray(X_train[idx_iso])

start = time.time()
iso = IsolationForest(
    n_estimators=250,
    contamination="auto",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
iso.fit(X_iso)

iso_raw = iso.predict(np.asarray(X_val))  # +1 normal, -1 anomaly
iso_attack = (iso_raw == -1).astype(int)

rows.append(bin_metrics(
    "Isolation Forest", as_attack(y_val), iso_attack,
    'contamination="auto"'
))
joblib.dump(iso, M / "isolation_forest.joblib")

['F:\\SentinelAI\\models\\isolation_forest.joblib']

In [4]:
# DBSCAN does not have native predict(). Fit on an unlabeled TRAIN sample,
# then use nearest core point within eps as a conservative out-of-sample approximation.
n_db_train = min(15_000, len(X_train))
idx_db_train = rng.choice(len(X_train), size=n_db_train, replace=False)
X_db_train = np.asarray(X_train[idx_db_train])

pca = PCA(n_components=min(10, X_db_train.shape[1]), random_state=RANDOM_STATE)
Z_train = pca.fit_transform(X_db_train)

min_samples = 10
nn = NearestNeighbors(n_neighbors=min_samples, n_jobs=-1).fit(Z_train)
distances, _ = nn.kneighbors(Z_train)
kdist = np.sort(distances[:, -1])
eps = float(np.quantile(kdist, 0.95))

dbscan = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1)
dbscan.fit(Z_train)

# Evaluate on a separate validation sample (features only used for transform/predict approximation).
n_db_val = min(15_000, len(X_val))
idx_db_val = rng.choice(len(X_val), size=n_db_val, replace=False)
Z_val = pca.transform(np.asarray(X_val[idx_db_val]))

if len(dbscan.core_sample_indices_) == 0:
    db_attack = np.ones(n_db_val, dtype=int)
    db_method = f"no_core_points; eps={eps:.4f}"
else:
    core_Z = Z_train[dbscan.core_sample_indices_]
    core_labels = dbscan.labels_[dbscan.core_sample_indices_]
    core_nn = NearestNeighbors(n_neighbors=1, n_jobs=-1).fit(core_Z)
    d, nearest = core_nn.kneighbors(Z_val)
    d = d[:, 0]
    nearest = nearest[:, 0]
    approx_cluster = np.where(d <= eps, core_labels[nearest], -1)
    db_attack = (approx_cluster == -1).astype(int)
    db_method = f"nearest_core_out_of_sample; eps={eps:.4f}; min_samples={min_samples}"

rows.append(bin_metrics(
    "DBSCAN", as_attack(np.asarray(y_val[idx_db_val])), db_attack, db_method
))
joblib.dump(dbscan, M / "dbscan.joblib")
joblib.dump(pca, M / "dbscan_pca.joblib")

print("DBSCAN eps:", eps)
print("Clusters:", len(set(dbscan.labels_)) - (1 if -1 in dbscan.labels_ else 0))
print("Noise ratio (train sample):", float(np.mean(dbscan.labels_ == -1)))

DBSCAN eps: 1.0942305734031306
Clusters: 21
Noise ratio (train sample): 0.039266666666666665


In [5]:
summary = pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)
summary.to_csv(M / "unsupervised_comparison.csv", index=False)
display(summary)

,Algorithm,Task,Evaluation_Split,Accuracy,Precision,Recall,F1,Label_Use,Method
0,K-Means,binary_anomaly_detection,validation,0.585683,0.165365,0.208107,0.184290,post_hoc_evaluation_only,minority_cluster=1; train_cluster_sizes=[71802...
1,Isolation Forest,binary_anomaly_detection,validation,0.685074,0.018738,0.007793,0.011008,post_hoc_evaluation_only,"contamination=""auto"""
2,DBSCAN,binary_anomaly_detection,validation,0.734600,0.029008,0.005648,0.009455,post_hoc_evaluation_only,nearest_core_out_of_sample; eps=1.0942; min_sa...


### تفسير صحيح للمقارنة

مقاييس Unsupervised هنا تخص **Benign vs Attack (binary anomaly detection)**، وليست نفس مهمة
التصنيف متعدد الفئات في Supervised/CNN. لذلك لا نستخدم F1 الثنائي لاختيار أفضل multiclass classifier.